# Windows Security Log Demo

Simple, non-streaming demo: read the JSON file produced by `winlog_demo.ps1`
and show it as a table. No Auto Loader, no checkpoints, no streaming — just
a batch read, a couple of summaries, and an optional table save.

Edit the two widgets below to point at a catalog/schema/volume you already
have access to — no new catalog or schema needs to be created for this.
The file itself can arrive either via `databricks fs cp` or by dragging it
into the volume through the Databricks UI.

In [0]:
dbutils.widgets.text("volume_path", "/Volumes/staging_monthly/winlog/landing", "Volume path (edit me)")
dbutils.widgets.text("table_name", "staging_monthly.winlog.security_demo", "Destination table (edit me)")

In [0]:
volume_path = dbutils.widgets.get("volume_path")
table_name  = dbutils.widgets.get("table_name")
file_path   = f"{volume_path}/winlog_demo_security_log.json"

### Read the file
One JSON object per line (newline-delimited JSON) — Spark reads this shape natively.

In [0]:
df = spark.read.json(file_path)
display(df)

### A couple of quick summaries
The kind of rollup a Splunk/qmulos dashboard would show.

In [0]:
display(df.groupBy("level").count().orderBy("count", ascending=False))

In [0]:
display(df.groupBy("event_id", "task_display_name").count().orderBy("count", ascending=False))

### Save as a table (optional)
Plain batch overwrite — no merge/upsert logic needed for a one-off demo.

In [0]:
df.write.mode("overwrite").saveAsTable(table_name)
display(spark.sql(f"SELECT * FROM {table_name} LIMIT 20"))

# Create the audit log table

This is the table the app writes to.

In [0]:
%sql
CREATE TABLE IF NOT EXISTS staging_monthly.winlog.nist_audit_log (
    audit_timestamp TIMESTAMP,
    nist_control STRING,
    status STRING,
    auditor STRING
  );

# Grant access to app

Compute → Apps → your app → App ID - there's an "Authorization"/permissions section that shows the app's service principal (a name like app-<something> plus an application/client ID). That's the identity app.py's WorkspaceClient() runs as when it hits the SQL warehouse.

In [0]:
%sql
GRANT USE CATALOG ON CATALOG staging_monthly TO `73154cb3-652a-42b3-9aaf-00330465bfc4`;
GRANT USE SCHEMA ON SCHEMA staging_monthly.winlog TO `73154cb3-652a-42b3-9aaf-00330465bfc4`;
GRANT SELECT, MODIFY ON TABLE staging_monthly.winlog.nist_audit_log TO `73154cb3-652a-42b3-9aaf-00330465bfc4`;
GRANT SELECT ON TABLE staging_monthly.winlog.security_demo TO `73154cb3-652a-42b3-9aaf-00330465bfc4`;